In [ ]:
#raw data folders
RAW_TAXI_FOLDER = "raw_data/to_processed/taxi"
RAW_WEATHER_FOLDER = "raw_data/to_processed/weather"

#Dim data 
DIM_PAYMENT_TYPE_FILE_PATH = "transformed_data/dim_payment_type/dim_payment_type.csv"
DIM_COMPANY_FILE_PATH = "transformed_data/dim_company/dim_company.csv"

In [ ]:
import pandas as pd
from io import StringIO
import json
from typing import Dict, List


def read_file_from_s3(s3, bucket:str, key: str, file_format:str ="csv"):
    #---    
    #Reads a csv or json file from an S3 bucket

    #:param s3:         S3 cliebt
    #:param bucket:     name of the S3 bucket where the file is stored
    #:param key:        Path within the S3 bucket
    #:file_format:      json or csv
    #---
    response = s3.get_object(Bucket=bucket, Key=key)
    content = response['Body'].read().decode('utf-8')

    if file_format == 'csv':
        return pd.read_csv(StringIO(content))
    elif file_format == 'json':
        return json.loads(content)
    else:
        raise ValueError("Unsupported file format. Use 'csv' or 'json'.")        


def transform_taxi(raw_taxi_data: List[Dict]) -> pd.DataFrame:
    #---
    #performs transformation on taxi data

    #1. Drop selected columns
    #2. Drop NULL values across columns
    #3. Rename columns
    #4. Create datetime_for_weather helper column (for dim_weather join)
    #
    #:param raw_taxi_data: json file holding daily taxi trips
    #:return:           transformed taxi trips dataframe
    #---

    taxi_trips= pd.DataFrame(raw_taxi_data)

    taxi_trips.drop(["pickup_census_tract","dropoff_census_tract","pickup_centroid_location",
                 "dropoff_centroid_location"],axis=1,inplace=True)
    taxi_trips.dropna(inplace=True)   #ha valamelyik rekordban Nan van akkor az egész rekordot törli (adattisztítás)

    taxi_trips.rename(columns={"pickup_community_area":"pickup_community_area_id",
                           "dropoff_community_area":"dropoff_community_area_id"},inplace=True)

    taxi_trips["trip_start_timestamp"]=pd.to_datetime(taxi_trips["trip_start_timestamp"])

    taxi_trips["datetime_for_weather"]=taxi_trips["trip_start_timestamp"].dt.floor("h")   

    return taxi_trips
    
def transform_weather(data: Dict) -> pd.DataFrame:
    #--
    # Select and transform weather data
    #:param data:       daily weather data from openmeteo API
    #:return:                   transformed weather pandas dataframe
    #---

    weather_data ={
        "datetime" : data["hourly"]["time"],
        "temperature" : data["hourly"]["temperature_2m"],
        "wind_speed" : data["hourly"]["wind_speed_10m"],
        "rain" : data["hourly"]["rain"],
        "precipitation" : data["hourly"]["precipitation"]
    }

    weather_df = pd.DataFrame(weather_data)
    weather_df["datetime"] = pd.to_datetime(weather_df["datetime"] )

    return weather_df


def update_dim_table(taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    #---Extend the dimenion dataframe with new value if has any (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_df:             dataframe dimension data (company, payment)
    #:param value_col           name of column of dimension dataframe contining values
    #:return: extended dimension data
    #---

    id_col=f"{value_col}_id"

    todays_dim_data=pd.DataFrame(taxi_trips[value_col].unique(),columns=[value_col])

    new_dim_data=todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    if not new_dim_data.empty:
        max_id=dim_df[id_col].max()
        new_dim_data[id_col]=range(max_id+1,max_id+1+len(new_dim_data))
        dim_df=pd.concat([dim_df,new_dim_data],ignore_index=True)

    return dim_df

def update_fact_taxi_trips_with_dim_data(taxi_trips: pd.DataFrame, dim_payment_type: pd.DataFrame,dim_company: pd.DataFrame) ->pd.DataFrame:
    #--
    # Update fact_taxi_trips Dataframe with the dim_company and dim_payment_type ids and delete the string columns (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_payment_type:    payment type master table
    #:param dim_company:         company master table
    #:return: taxi trips data with ids without company and payment type values
    #---
    fact_taxi_trips=taxi_trips.merge(dim_payment_type,on="payment_type")
    fact_taxi_trips=fact_taxi_trips.merge(dim_company,on="company")
    fact_taxi_trips.drop(["payment_type","company"],axis=1,inplace=True)

    return fact_taxi_trips


def _move_file_on_s3(s3, bucket:str, source_key:str, target_key:str):
#move the files within s3
    s3.copy_object(Bucket=bucket, CopySource={"Bucket":bucket, "Key": source_key}, Key=target_key)
    s3.delete_object(Bucket=bucket, Key=source_key)

    print(f"Archived raw data.")


def _upload_dataframe_to_s3(s3, bucket:str, dataframe: pd.DataFrame, path: str):
    #---    
    #Uploads a dataframe to the specified S3 path

    #:param s3:         S3 cliebt
    #:param dataframe:  name of the dataframe to be upladed
    #:param bucket:     name of the S3 bucket where the file will be stored
    #:param path:       path within the  bucket to upload the file
    #---

    buffer = StringIO()
    dataframe.to_csv(buffer, index=False)
    df_content = buffer.getvalue()
    s3.put_object(Bucket=bucket, Key=path, Body=df_content)

    print("Uploaded dataframe to S3")


def upload_dim_to_s3(s3, bucket: str, dim_type:str, dataframe:pd.DataFrame):
    #---    
    #Uploads a dimesion table (e.g. company od payment_type) to S3 
    #copies the previous version before overwriting the current one

    #:param s3:             S3 cliebt
    #:param bucket:         name of the S3 bucket
    #:param dim_type:       name of the the dimension (e.g. "company" or "payment_type")
    #:param dataframe:      dataframe to uplad
    #:raises ValueError:    Raises when file_type is not "company" or "payment_type"
    #---

    if dim_type not in ["company", "payment_type"]:
        raise ValueError("Unsupported dimension type. Use 'company' or 'payment_type'.")
    #---

    current_file_path=f"transformed_data/dim_{dim_type}/dim_{dim_type}.csv"
    previous_version_file_path=f"transformed_data/dimension_table_previous_versions/dim_{dim_type}.csv"

    s3.copy_object(Bucket=bucket, CopySource={"Bucket":bucket, "Key": current_file_path}, Key=previous_version_file_path)

    print(f"Copied existed version {dim_type} to previous version folder")

    _upload_dataframe_to_s3(s3, bucket, dataframe, path=current_file_path)


def upload_and_archive_on_s3(s3, dataframe: pd.DataFrame, bucket: str,  file_type: str):
#Uploads a transformed dataset to S3 and achives the corresponig raw file
#
#Workflow:
#1. Uploads a transformed Dataframe to its folder with a date-based filename
#2. Moves the original raw file from to_processed to processed
    #:param s3:             S3 cliebt
    #:param dataframe:      tarnsformed Dataframe to uplad
    #:param bucket:         name of the S3 bucket
    #:param file_type:      type of the file (e.g. "taxi" or "weather")
    
    match file_type:
        case "taxi": 
            formatted_date = dataframe["datetime_for_weather"].dt.strftime("%Y-%m-%d").iloc[0]
            transformed_key = f"transformed_data/fact_taxi_trips/taxi_{formatted_date}.csv"
        case "weather":
            formatted_date = dataframe["datetime"].dt.strftime("%Y-%m-%d").iloc[0]
            transformed_key = f"transformed_data/dim_weather/weather_{formatted_date}.csv"
        case _:
            raise ValueError("Unsupported file type. Use 'taxis' or 'weather'.")

    #1. upload transformed data
    _upload_dataframe_to_s3(s3, bucket, dataframe, transformed_key)

    #2. move raw file to archive
    source__key = f"raw_data/to_processed/{file_type}/{file_type}_{formatted_date}.json"
    target_key = f"raw_data/processed/{file_type}/{file_type}_{formatted_date}.json"

    _move_file_on_s3(s3, bucket, source__key, target_key)

    

In [ ]:
import boto3
import json
from typing import Dict

import pandas as pd

from configs import (
   BUCKET,
   DIM_COMPANY_FILE_PATH,
   DIM_PAYMENT_TYPE_FILE_PATH,
   RAW_TAXI_FOLDER,
   RAW_WEATHER_FOLDER
)

from functions import (
   read_file_from_s3,
   transform_taxi,
   transform_weather,
   update_dim_table,
   update_fact_taxi_trips_with_dim_data,
   upload_and_archive_on_s3,
   upload_dim_to_s3
)


def process_weather_data(s3):
   """Process and transform daily weather data

   1. Download the raw weather data
   2. Read and transform it
   3. Upload and archive

   """
   for file in s3.list_objects(Bucket=BUCKET, Prefix=RAW_WEATHER_FOLDER)["Contents"]:
      weather_key=file["Key"]

      weather_raw_filename=file["Key"].split("/")[-1]
      if weather_raw_filename.split(".")[-1] == "json":
         weather_raw_content = read_file_from_s3(s3, BUCKET, weather_key, "json")

         dim_weather = transform_weather(weather_raw_content)

         upload_and_archive_on_s3(s3, dim_weather, BUCKET, "weather")



def process_taxi_data(s3, dim_payment_type: pd.DataFrame, dim_company: pd.DataFrame):
   """Process and transform daily taxi data

   1. Download the raw taxi data
   2. Read and transform it
   3. Update the payment type and company tables
   4. Update taxi data with payment type and company ids
   5. Upload payment type and company to S3
   6. Archive taxi data

   :param s3:                 S3 client
   :param dim_payment_type:   Payment type dimension table
   :param dim_company:        Company dimension table

   """
   for file in s3.list_objects(Bucket=BUCKET, Prefix=RAW_TAXI_FOLDER)["Contents"]:
      taxi_key=file["Key"]
      taxi_raw_filename=file["Key"].split("/")[-1]

      if taxi_raw_filename.split(".")[-1] == "json":
         taxi_raw_content = read_file_from_s3(s3, BUCKET, taxi_key, "json")

         fact_taxi_trips = transform_taxi(taxi_raw_content)

         dim_payment_type_updated= update_dim_table(fact_taxi_trips, dim_payment_type,"payment_type")
         dim_company_updated = update_dim_table(fact_taxi_trips, dim_company, "company")

         fact_taxi_trips_updated = update_fact_taxi_trips_with_dim_data(fact_taxi_trips, dim_payment_type, dim_company)

         upload_dim_to_s3(s3, BUCKET, "payment_type", dim_payment_type_updated)
         upload_dim_to_s3(s3, BUCKET, "company", dim_company_updated)

         upload_and_archive_on_s3(s3, fact_taxi_trips_updated, BUCKET, "taxi")



def lambda_handler(event, context):
   s3=boto3.client('s3')

   dim_payment_type=read_file_from_s3(s3, BUCKET, DIM_PAYMENT_TYPE_FILE_PATH, "csv")
   dim_company=read_file_from_s3(s3, BUCKET, DIM_COMPANY_FILE_PATH, "csv")

   #TAXI transformation
   process_taxi_data(s3, dim_payment_type, dim_company)
   # #WEATHER transformation
   process_weather_data(s3)

   print("Done")
